# PDF zu Markdown (Batch)

Dieses Notebook konvertiert PDF-Dateien in `.md`-Dateien.
- Unterstuetzt einzelne Datei oder kompletten Ordner
- Optional rekursiv mit Unterordnern
- Optional Seitenlimit pro PDF

In [ ]:
# Falls pypdf fehlt, diese Zeile einmal ausfuehren:
#%pip install pypdf

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
from pathlib import Path
from pypdf import PdfReader

INPUT_PATH = Path(r"C:/Users/Felix/Desktop/Master/Thesis/Quellen/ML")
OUTPUT_PATH = Path(r"C:/Users/Felix/Desktop/Master/Thesis/touch-scrolling-web-app/thesis/quellen")
RECURSIVE = True
KEEP_STRUCTURE = True
MAX_PAGES = None  # z. B. 30 fuer nur erste 30 Seiten
ADD_SOURCE_HEADER = True
OVERWRITE_EXISTING = False  # False: vorhandene .md-Dateien werden uebersprungen

OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

In [15]:
def find_pdf_files(input_path: Path, recursive: bool):
    if input_path.is_file() and input_path.suffix.lower() == '.pdf':
        return [input_path]

    if not input_path.is_dir():
        return []

    pattern = '**/*.pdf' if recursive else '*.pdf'
    return sorted(input_path.glob(pattern))


def extract_text_from_pdf(pdf_path: Path, max_pages=None) -> str:
    reader = PdfReader(str(pdf_path))
    total_pages = len(reader.pages)
    pages_to_read = total_pages if max_pages is None else min(max_pages, total_pages)

    chunks = []
    for i in range(pages_to_read):
        text = reader.pages[i].extract_text() or ''
        chunks.append(text)

    return '\n\n'.join(chunks)


def build_markdown_content(pdf_path: Path, extracted_text: str, add_source_header: bool) -> str:
    if not add_source_header:
        return extracted_text

    header = [
        f"# {pdf_path.stem}",
        '',
        f"Quelle: {pdf_path}",
        '',
        '---',
        '',
    ]
    return '\n'.join(header) + extracted_text


def build_output_path(pdf_path: Path, input_path: Path, output_root: Path, keep_structure: bool) -> Path:
    if keep_structure and input_path.is_dir():
        rel = pdf_path.relative_to(input_path).with_suffix('.md')
        return output_root / rel
    return output_root / f'{pdf_path.stem}.md'

In [16]:
pdf_files = find_pdf_files(INPUT_PATH, recursive=RECURSIVE)

if not pdf_files:
    raise FileNotFoundError(f'Keine PDFs gefunden unter: {INPUT_PATH}')

converted = 0
skipped = 0
failed = 0

for pdf_file in pdf_files:
    try:
        out_file = build_output_path(pdf_file, INPUT_PATH, OUTPUT_PATH, KEEP_STRUCTURE)

        if out_file.exists() and not OVERWRITE_EXISTING:
            skipped += 1
            print(f'SKIP: {pdf_file.name} -> {out_file} (bereits vorhanden)')
            continue

        text = extract_text_from_pdf(pdf_file, max_pages=MAX_PAGES)
        markdown_content = build_markdown_content(
            pdf_path=pdf_file,
            extracted_text=text,
            add_source_header=ADD_SOURCE_HEADER,
        )

        out_file.parent.mkdir(parents=True, exist_ok=True)
        out_file.write_text(markdown_content, encoding='utf-8')
        converted += 1
        print(f'OK: {pdf_file.name} -> {out_file}')
    except Exception as exc:
        failed += 1
        print(f'ERROR: {pdf_file} | {exc}')

print(f'Fertig. Konvertiert: {converted}, Uebersprungen: {skipped}, Fehler: {failed}')

OK: Active Learning and Bayesian Optimization A Unified Perspective.pdf -> C:\Users\Felix\Desktop\Master\Thesis\touch-scrolling-web-app\thesis\quellen\Active Learning and Bayesian Optimization A Unified Perspective.md
OK: Active_Learning_Literature_Survey.pdf -> C:\Users\Felix\Desktop\Master\Thesis\touch-scrolling-web-app\thesis\quellen\Active_Learning_Literature_Survey.md


Ignoring wrong pointing object 43 0 (offset 0)
Ignoring wrong pointing object 45 0 (offset 0)
Ignoring wrong pointing object 48 0 (offset 0)


OK: ActiveLearning.pdf -> C:\Users\Felix\Desktop\Master\Thesis\touch-scrolling-web-app\thesis\quellen\ActiveLearning.md
OK: BayesianActiveLearningWithFullyBayesianGaussianProcesses.pdf -> C:\Users\Felix\Desktop\Master\Thesis\touch-scrolling-web-app\thesis\quellen\BayesianActiveLearningWithFullyBayesianGaussianProcesses.md
OK: bo_torch_paper.pdf -> C:\Users\Felix\Desktop\Master\Thesis\touch-scrolling-web-app\thesis\quellen\bo_torch_paper.md
OK: gaussian-processes-for-regression.pdf -> C:\Users\Felix\Desktop\Master\Thesis\touch-scrolling-web-app\thesis\quellen\gaussian-processes-for-regression.md
OK: Gaussian_processes_for_machine_learning.pdf -> C:\Users\Felix\Desktop\Master\Thesis\touch-scrolling-web-app\thesis\quellen\Gaussian_processes_for_machine_learning.md
OK: MonteCarlo.pdf -> C:\Users\Felix\Desktop\Master\Thesis\touch-scrolling-web-app\thesis\quellen\MonteCarlo.md
OK: Multi Task Learning With Gaussian Processes.pdf -> C:\Users\Felix\Desktop\Master\Thesis\touch-scrolling-web-app\